# vLLM-Hook Minimal Parity: Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/vllm-hook-mlx/tests/parity_tests/colab_minimal_parity.ipynb)

This notebook runs the Colab/non-Metal half of `tests/parity_tests/run_all_minimal_parity.py` directly inside a Colab GPU runtime. It is intended as a manual replacement while the local `colab` CLI workflow is unavailable.

Run the Mac/Metal side locally with the same `BENCHMARK_PREFIX`, then run this notebook on Colab with a GPU runtime. Matching runs use W&B groups named `<BENCHMARK_PREFIX>-<experiment>`.

## 1. Bootstrap Repo And Dependencies

Set `REPO_URL` and `REPO_BRANCH` to a branch that contains `tests/parity_tests/minimal_parity_benchmarks.py`. On Colab, this cell clones or refreshes the repo, installs requirements, installs the plugin package editable, and verifies CUDA is available.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "vllm-hook-mlx")
WORKDIR = Path(os.environ.get("VLLM_HOOK_COLAB_WORKDIR", "/content/vLLM-Hook"))
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")

def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("+ " + " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        tail = tail[-80:]
    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )

def run_shell(command, cwd=None, env=None):
    print("+ " + command, flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, shell=True, executable="/bin/bash", check=True)

def repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return origin_url == expected_remote

def prepare_repo(repo_url: str, branch: str, workdir: Path) -> Path:
    if not repo_url:
        if workdir.exists():
            return workdir
        raise FileNotFoundError(f"Repo URL is empty and {workdir} does not exist.")
    expected_remote = repo_url.removesuffix(".git")
    if not workdir.exists():
        run(["git", "clone", "--branch", branch, repo_url, workdir])
    elif not repo_remote_matches(workdir, expected_remote):
        print(f"Remote mismatch under {workdir}; replacing clone with {expected_remote}")
        shutil.rmtree(workdir)
        run(["git", "clone", "--branch", branch, repo_url, workdir])
    else:
        print(f"Reusing existing clone at {workdir}")
    run(["git", "-C", workdir, "fetch", "origin", branch])
    run(["git", "-C", workdir, "checkout", branch])
    run(["git", "-C", workdir, "pull", "--ff-only", "origin", branch])
    return workdir

def assert_cuda_runtime():
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError("Choose Runtime > Change runtime type > GPU, then rerun from a fresh runtime.")

PROJECT_ROOT = prepare_repo(REPO_URL, REPO_BRANCH, WORKDIR)
os.chdir(PROJECT_ROOT)
assert_cuda_runtime()

plugin_dir = PROJECT_ROOT / "vllm_hook_plugins"
req = PROJECT_ROOT / "requirement.txt"
if not plugin_dir.exists():
    raise FileNotFoundError(f"Plugin directory not found: {plugin_dir}")

run_shell(f"{sys.executable} -m pip install -U pip")
if req.exists():
    run_shell(f"{sys.executable} -m pip install -r {req}")
else:
    print("Warning: requirement.txt not found; skipping dependency install.")
run_shell(f"{sys.executable} -m pip install --force-reinstall 'protobuf>=5.29.6,<6.30'")
run_shell(f"{sys.executable} -m pip install wandb weave pytest")
if COLAB_INSTALL_VLLM:
    run_shell(f"{sys.executable} -m pip install {COLAB_INSTALL_VLLM}")
run_shell(f"{sys.executable} -m pip install -e {plugin_dir}")

plugin_src = str(plugin_dir.resolve())
if plugin_src not in sys.path:
    sys.path.insert(0, plugin_src)
importlib.invalidate_caches()

RUNNER = PROJECT_ROOT / "tests" / "parity_tests" / "minimal_parity_benchmarks.py"

try:
    import torch
    print("torch       :", torch.__version__)
    print("cuda        :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda device :", torch.cuda.get_device_name(0))
except Exception as exc:
    print("torch check failed:", type(exc).__name__, exc)

try:
    import vllm
    print("vllm        :", getattr(vllm, "__version__", "unknown"))
except Exception as exc:
    print("vllm check failed:", type(exc).__name__, exc)

print("Project root:", PROJECT_ROOT)
print("Runner      :", RUNNER)
print("Python      :", sys.executable)

## 2. Authenticate W&B And Hugging Face

Leave the constants blank to use environment variables, Colab Secrets, or prompts. The Hugging Face token is optional unless a selected model requires it.

In [ ]:
import getpass
import os

HARDCODED_WANDB_API_KEY = ""
HARDCODED_HF_TOKEN = ""

try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret_or_prompt(name: str, required: bool = False):
    hardcoded = {
        "WANDB_API_KEY": HARDCODED_WANDB_API_KEY,
        "HF_TOKEN": HARDCODED_HF_TOKEN,
    }.get(name)
    if hardcoded:
        os.environ[name] = hardcoded
        return hardcoded
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return value
    if required:
        value = getpass.getpass(f"{name}: ")
        os.environ[name] = value
        return value
    return None

secret_or_prompt("WANDB_API_KEY", required=True)
secret_or_prompt("HF_TOKEN", required=False)

if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

import wandb
wandb.login(key=os.environ.get("WANDB_API_KEY"), relogin=True)
print("W&B login configured")

## 3. Configure The Comparison Run

Use the same `BENCHMARK_PREFIX` as the local Mac run. The app-specific W&B projects are selected by `minimal_parity_benchmarks.py`; `WANDB_PROJECT` remains the fallback.

In [ ]:
EXPERIMENTS = ("attn-tracker", "core-reranker", "steer-activation")
MODELS = {
    "attn-tracker": os.environ.get("ATTN_TRACKER_MODEL", "Qwen/Qwen2-1.5B-Instruct"),
    "core-reranker": os.environ.get("CORE_RERANKER_MODEL", "mistralai/Mistral-7B-Instruct-v0.3"),
    "steer-activation": os.environ.get("STEER_ACTIVATION_MODEL", "microsoft/Phi-3-mini-4k-instruct"),
}

BENCHMARK_PREFIX = os.environ.get("BENCHMARK_PREFIX", "minimal-parity")
WANDB_MODE = os.environ.get("WANDB_MODE", "online")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "vllm-hook-platform-parity")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
HARDWARE_LABEL = os.environ.get("VLLM_HOOK_HARDWARE_LABEL", "colab-gpu")
HARDWARE_KIND = os.environ.get("VLLM_HOOK_HARDWARE_KIND", "cuda")
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "2"))
TEMPERATURE = 0.0
TOP_P = float(os.environ.get("TOP_P", "1.0"))
GPU_MEMORY_UTILIZATION = float(os.environ.get("GPU_MEMORY_UTILIZATION", "0.5"))
MAX_MODEL_LEN = int(os.environ.get("MAX_MODEL_LEN", "2048"))
DTYPE = os.environ.get("DTYPE", "float16")

SHARED_INPUTS = {
    "experiments": EXPERIMENTS,
    "models": MODELS,
    "benchmark_prefix": BENCHMARK_PREFIX,
    "max_tokens": MAX_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "max_model_len": MAX_MODEL_LEN,
    "dtype": DTYPE,
}

print("benchmark prefix:", BENCHMARK_PREFIX)
print("wandb mode     :", WANDB_MODE)
print("wandb project  :", WANDB_PROJECT)
print("wandb entity   :", WANDB_ENTITY or "<default>")
print("hardware label :", HARDWARE_LABEL)
print("shared inputs  :", SHARED_INPUTS)

## 4. Helper To Run A Colab Experiment

This uses the checked-in minimal parity runner with `--backend non-metal`, matching `colab_minimal_parity_remote.py`.

In [ ]:
def run_colab_parity(experiment: str):
    if experiment not in EXPERIMENTS:
        raise ValueError(f"Expected one of {EXPERIMENTS}, got {experiment!r}")
    env = os.environ.copy()
    env["WANDB_MODE"] = WANDB_MODE
    env["WANDB_PROJECT"] = WANDB_PROJECT
    env.setdefault("VLLM_USE_V1", "1")
    env.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
    if WANDB_ENTITY:
        env["WANDB_ENTITY"] = WANDB_ENTITY

    benchmark_id = f"{BENCHMARK_PREFIX}-{experiment}"
    cmd = [
        sys.executable,
        str(RUNNER),
        experiment,
        "--backend", "non-metal",
        "--benchmark-id", benchmark_id,
        "--model", MODELS[experiment],
        "--wandb-mode", WANDB_MODE,
        "--wandb-project", WANDB_PROJECT,
        "--hardware-label", HARDWARE_LABEL,
        "--hardware-kind", HARDWARE_KIND,
        "--max-tokens", str(MAX_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-p", str(TOP_P),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--dtype", DTYPE,
    ]
    if WANDB_ENTITY:
        cmd.extend(["--wandb-entity", WANDB_ENTITY])
    run(cmd, cwd=PROJECT_ROOT, env=env)
    return PROJECT_ROOT / "tests" / "experiment_runs" / "minimal_parity" / experiment / benchmark_id / "non-metal"

## 5. Run One Experiment

Start with attention tracker because it uses the smaller default model.

In [ ]:
run_dir = run_colab_parity("attn-tracker")

Run the remaining experiments when ready.

In [ ]:
# run_dir = run_colab_parity("core-reranker")
# run_dir = run_colab_parity("steer-activation")

Run all Colab/non-Metal experiments.

In [ ]:
# for experiment in EXPERIMENTS:
#     run_dir = run_colab_parity(experiment)

## 6. Inspect Local Outputs

The runner writes local JSON, CSV, manifest, and hook artifacts under `tests/experiment_runs/minimal_parity` before logging to W&B.

In [ ]:
import json

def show_local_outputs(run_dir: Path):
    print("Run dir:", run_dir)
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            print(" ", path.relative_to(run_dir), path.stat().st_size, "bytes")
    manifest_path = run_dir / "artifact_manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        print("\nManifest files:", len(manifest.get("files", [])))
        for row in manifest.get("files", []):
            if row.get("path", "").endswith((".pt", ".safetensors")):
                print(row["path"], row.get("bytes"), row.get("sha256"))

show_local_outputs(run_dir)

## 7. Optional Artifact Lookup

Use this to find the latest W&B artifact for one experiment in the app-specific project.

In [ ]:
APP_PROJECTS = {
    "attn-tracker": "attntracker",
    "core-reranker": "corereranker",
    "steer-activation": "steering",
}

def download_latest_artifact(experiment="attn-tracker"):
    api = wandb.Api()
    entity = WANDB_ENTITY or api.default_entity
    project = APP_PROJECTS.get(experiment, WANDB_PROJECT)
    runs = api.runs(f"{entity}/{project}", order="-created_at", per_page=100)
    run = next(
        r for r in runs
        if experiment in r.tags and "non-metal" in r.tags and "minimal-parity" in r.tags
    )
    artifact = next(a for a in run.logged_artifacts() if a.type == "vllm-hook-minimal-parity")
    artifact_dir = Path(artifact.download())
    print("Run:", run.name, run.url)
    print("Artifact:", artifact.name)
    print("Downloaded to:", artifact_dir)
    return artifact_dir

# artifact_dir = download_latest_artifact("attn-tracker")